# Práctica 2 · Paso 1 — Inventario simplificado de emisiones de la provincia

**Cambio Climático · Ingeniería Ambiental · ESPAM MFL** · Material CC-IAM-P2-01 · Versión 1.0 · noviembre de 2026

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crispaliz/cambio-climatico-espam/blob/main/practicas/ambiental/practica-2/IA2_01_inventario_emisiones.ipynb)

**Objetivo.** Estimar las emisiones anuales de gases de efecto invernadero (GEI) de su provincia por sector y por gas, con el método del IPCC, e identificar sus categorías principales.

**Producto.** Tabla de emisiones por categoría y gas, resumen por sector, categorías principales, incertidumbre, comparación con el inventario nacional y ficha de datos de actividad con sus fuentes. Primera pieza del capítulo 2.

**Método.** Directrices del IPCC de 2006, Nivel 1 (factores por defecto). CO₂ equivalente con los potenciales de calentamiento global a 100 años del AR5 (CH₄ = 28; N₂O = 265), los mismos que usa el Inventario Nacional del Ecuador.

**Cómo trabajar.**
1. En Colab: **Archivo › Guardar una copia en Drive**.
2. **Antes de clase o durante ella**, reúna los datos de actividad de su provincia con la tabla de fuentes de la sección 3 y anote en la ficha de dónde sale cada número.
3. Escriba los datos en los formularios de las celdas 4 a 7 y ejecute todo en orden. Un dato que no encuentre se deja en 0: el cuaderno lo marca como **no estimado (NE)**, que es lo correcto en un inventario.

**Sin conexión o con energía intermitente.** El cálculo no descarga datos: si ya clonó el repositorio, funciona sin internet. La lectura de los datos de actividad se puede hacer en papel con la misma ficha.

## 1. Datos del estudiante

In [ ]:
# @title Datos del estudiante y provincia (la misma de la Práctica 1)
ESTUDIANTE = "Nombre Apellido"  # @param {type:"string"}
PROVINCIA = "Manabí"  # @param ["Azuay", "Bolívar", "Cañar", "Carchi", "Chimborazo", "Cotopaxi", "El Oro", "Esmeraldas", "Galápagos", "Guayas", "Imbabura", "Loja", "Los Ríos", "Manabí", "Morona Santiago", "Napo", "Orellana", "Pastaza", "Pichincha", "Santa Elena", "Santo Domingo de los Tsáchilas", "Sucumbíos", "Tungurahua", "Zamora Chinchipe"]
ANIO_INVENTARIO = 2024  # @param {type:"integer"}

## 2. Entorno de trabajo

In [ ]:
import os, sys, pathlib, subprocess, datetime, json
REPO_URL = "https://github.com/crispaliz/cambio-climatico-espam"
REPO_DIR = pathlib.Path(os.environ.get("CC_REPO", "/content/cambio-climatico-espam"))
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=False, stderr=subprocess.DEVNULL)
sys.path.insert(0, str(REPO_DIR))

from herramientas import inventario as inv
from herramientas.limites import slug

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = pathlib.Path("/content/drive/MyDrive")
except Exception:
    BASE = pathlib.Path(os.environ.get("CC_BASE", "."))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
S = slug(PROVINCIA)
RES = BASE / "CC_Ambiental" / "practica-2" / S
RES.mkdir(parents=True, exist_ok=True)
POB = inv.poblacion(PROVINCIA)
print(f"Provincia: {PROVINCIA} · población (Censo 2022): {POB:,} habitantes".replace(",", " "))
print(f"Año del inventario: {ANIO_INVENTARIO} · resultados en {RES}")

## 3. Datos de actividad: qué buscar y dónde

Emisión = **dato de actividad** × **factor de emisión**. Los factores ya están en el cuaderno (IPCC 2006, Nivel 1); **los datos de actividad los busca usted**. Use el año más reciente disponible para todos y anótelo.

| Sector IPCC | Dato de actividad | Unidad | Fuente sugerida |
|---|---|---|---|
| 1. Energía · transporte | Ventas de gasolina (extra y súper) y diésel en la provincia | galones/año | ARCERNNR, estadísticas de comercialización de combustibles por provincia; Balance Energético Nacional |
| 1. Energía · residencial | GLP de uso doméstico vendido en la provincia | kg/año (cilindro = 15 kg) | ARCERNNR, comercialización de GLP por provincia |
| Memo · electricidad | Energía eléctrica facturada en la provincia | MWh/año | ARCONEL, estadística del sector eléctrico (energía facturada por provincia) |
| 3. Agricultura · ganado | Cabezas de vacunos (vacas de ordeño aparte, si se reportan), porcinos, ovinos, caprinos, caballos, mulas y asnos | cabezas | INEC, Encuesta de Superficie y Producción Agropecuaria Continua (ESPAC) |
| 3. Agricultura · arroz | Superficie cosechada de arroz y régimen de riego | ha/año | INEC, ESPAC; MAG, SIPA (boletines provinciales) |
| 3. Agricultura · suelos | Nitrógeno sintético aplicado (urea tiene 46 % de N) | t de N/año | MAG, SIPA; estimación: superficie × dosis típica del cultivo |
| 4. USCUSS | Deforestación bruta anual y carbono del estrato de bosque | ha/año; t C/ha | MAATE, mapas de deforestación del Ecuador continental; Nivel de Referencia de Emisiones Forestales (NREF) del Ecuador |
| 5. Residuos · sólidos | Generación por habitante y tipo de sitio de disposición | kg/hab/día | INEC, Gestión de residuos sólidos en los GAD municipales; AME |
| 5. Residuos · aguas | Fracción de hogares por tipo de servicio higiénico | fracción | INEC, Censo de Población y Vivienda 2022 (servicio higiénico por provincia) |

La población ya está cargada (INEC, Censo 2022). Si un dato es una estimación suya, dígalo en la ficha.

**El inventario no incluye:** procesos industriales (PIUP, por ejemplo una cementera), aviación y navegación, combustibles de la industria y de la generación eléctrica local, N₂O del estiércol y de la orina en pasturas, quema de biomasa y remociones de bosques en pie. Son omisiones que debe declarar.

In [ ]:
# @title 4. Energía (ventas anuales en la provincia)
GASOLINA_GAL = 0  # @param {type:"number"}
DIESEL_GAL = 0  # @param {type:"number"}
GLP_KG = 0  # @param {type:"number"}
ELECTRICIDAD_MWH = 0  # @param {type:"number"}
FUENTE_ENERGIA = "ARCERNNR (año); ARCONEL (año)"  # @param {type:"string"}

In [ ]:
# @title 5. Agricultura (ganado en cabezas; arroz en ha cosechadas; nitrógeno en t de N)
VACAS_ORDENO = 0  # @param {type:"number"}
OTROS_VACUNOS = 0  # @param {type:"number"}
PORCINOS = 0  # @param {type:"number"}
OVINOS = 0  # @param {type:"number"}
CAPRINOS = 0  # @param {type:"number"}
CABALLOS = 0  # @param {type:"number"}
MULAS_ASNOS = 0  # @param {type:"number"}
ARROZ_HA = 0  # @param {type:"number"}
ARROZ_DIAS = 120  # @param {type:"integer"}
ARROZ_REGIMEN = "inundación continua"  # @param ["inundación continua", "inundación intermitente (un drenaje)", "inundación intermitente (varios drenajes)", "secano (lluvia)"]
N_SINTETICO_T = 0  # @param {type:"number"}
FUENTE_AGRICULTURA = "INEC, ESPAC (año)"  # @param {type:"string"}
# Si la ESPAC da solo el total de vacunos, póngalo todo en OTROS_VACUNOS y dígalo en el informe.

In [ ]:
# @title 6. USCUSS (deforestación)
DEFORESTACION_HA = 0  # @param {type:"number"}
CARBONO_TC_HA = 0  # @param {type:"number"}
FUENTE_USCUSS = "MAATE, mapa de deforestación (periodo); NREF (estrato)"  # @param {type:"string"}
# CARBONO_TC_HA: carbono de biomasa del estrato de bosque de su provincia según el NREF del Ecuador.

In [ ]:
# @title 7. Residuos
GEN_KG_HAB_DIA = 0  # @param {type:"number"}
FRAC_DISPUESTA = 0.9  # @param {type:"number"}
SITIO = "sitio no clasificado"  # @param ["relleno sanitario gestionado", "botadero profundo (> 5 m)", "botadero poco profundo (< 5 m)", "sitio no clasificado"]
# Fracción de la población por tipo de servicio higiénico (deben sumar 1; todo en 0 = no estimado)
ALC_PLANTA_AEROBIA = 0.0  # @param {type:"number"}
ALC_LAGUNA_ANAEROBIA = 0.0  # @param {type:"number"}
ALC_SIN_TRATAMIENTO = 0.0  # @param {type:"number"}
POZO_SEPTICO = 0.0  # @param {type:"number"}
LETRINA = 0.0  # @param {type:"number"}
SIN_SERVICIO = 0.0  # @param {type:"number"}
PROTEINA_KG_HAB_ANIO = 25  # @param {type:"number"}
FUENTE_RESIDUOS = "INEC, GAD municipales (año); Censo 2022"  # @param {type:"string"}

## 8. Cálculo del inventario

In [ ]:
AGUAS = {"alcantarillado con planta aerobia": ALC_PLANTA_AEROBIA, "alcantarillado con laguna anaerobia": ALC_LAGUNA_ANAEROBIA,
         "alcantarillado sin tratamiento (a río o mar)": ALC_SIN_TRATAMIENTO, "pozo séptico": POZO_SEPTICO,
         "letrina": LETRINA, "sin servicio (a campo abierto)": SIN_SERVICIO}
AGUAS = {k: v for k, v in AGUAS.items() if v} or None

filas = []
filas += inv.energia(gasolina_gal=GASOLINA_GAL, diesel_gal=DIESEL_GAL, glp_kg=GLP_KG, electricidad_mwh=ELECTRICIDAD_MWH)
filas += inv.agricultura(arroz_ha=ARROZ_HA, arroz_dias=ARROZ_DIAS, arroz_regimen=ARROZ_REGIMEN, n_sintetico_t=N_SINTETICO_T,
                         vacas_leche=VACAS_ORDENO, otros_vacunos=OTROS_VACUNOS, porcinos=PORCINOS, ovinos=OVINOS,
                         caprinos=CAPRINOS, caballos=CABALLOS, mulas_asnos=MULAS_ASNOS)
filas += inv.uscuss(DEFORESTACION_HA, CARBONO_TC_HA)
filas += inv.residuos(POB, GEN_KG_HAB_DIA, FRAC_DISPUESTA, SITIO, aguas=AGUAS, proteina_kg_hab_anio=PROTEINA_KG_HAB_ANIO)
TAB = inv.tabla(filas)

# Completitud: qué categorías quedaron sin estimar
CATEGORIAS = {"1A3b Transporte por carretera": GASOLINA_GAL or DIESEL_GAL, "1A4b Residencial (GLP)": GLP_KG,
              "3A1/3A2 Ganado": VACAS_ORDENO or OTROS_VACUNOS or PORCINOS or OVINOS or CAPRINOS or CABALLOS or MULAS_ASNOS,
              "3C7 Cultivo de arroz": ARROZ_HA, "3C4/3C5 Suelos (N sintético)": N_SINTETICO_T,
              "3B1b Deforestación": DEFORESTACION_HA and CARBONO_TC_HA, "4A Residuos sólidos": GEN_KG_HAB_DIA,
              "4D1 Aguas residuales": AGUAS, "Memo: electricidad": ELECTRICIDAD_MWH}
COMPLETITUD = pd.Series({k: "estimado" if v else "NE (no estimado)" for k, v in CATEGORIAS.items()}, name="estado")
display(COMPLETITUD.to_frame())
cols = ["sector", "categoria", "gas", "dato_actividad", "unidad_actividad", "factor", "unidad_factor", "emision_t_gas", "emision_tCO2e"]
display(TAB[cols].style.format({"dato_actividad": "{:,.0f}", "factor": "{:,.4g}", "emision_t_gas": "{:,.1f}", "emision_tCO2e": "{:,.0f}"}))

## 9. Resultados: sectores, gases, categorías principales e incertidumbre

In [ ]:
TOTAL = inv.total(TAB)
SECT = inv.resumen(TAB, "sector")
GAS = inv.resumen(TAB, "gas")
CLAVE = inv.categorias_principales(TAB)
U = inv.incertidumbre(TAB)
print(f"Total provincial {ANIO_INVENTARIO}: {TOTAL/1000:,.1f} kt CO2-eq  (± {U:.0f} %, enfoque 1 del IPCC)")
print(f"Por habitante: {TOTAL/POB:.2f} t CO2-eq/hab")
memo = TAB[TAB.sector.str.startswith("Memo")]
if not memo.empty:
    print(f"Memo, electricidad consumida (alcance 2, no sumada): {memo.emision_tCO2e.sum()/1000:,.1f} kt CO2-eq")
display(SECT.style.format({"t CO2-eq": "{:,.0f}", "%": "{:.1f}"}))
display(GAS.style.format({"t CO2-eq": "{:,.0f}", "%": "{:.1f}"}))
display(CLAVE.style.format({"t CO2-eq": "{:,.0f}", "%": "{:.1f}", "% acumulado": "{:.1f}"}))

In [ ]:
# Comparación con el inventario nacional (año 2022, Segunda NDC del Ecuador)
NAC = pd.Series(inv.NACIONAL_2022, name="nacional kt CO2-eq")
COMP = pd.DataFrame({"provincia kt CO2-eq": SECT["t CO2-eq"] / 1000}).join(NAC, how="outer").fillna(0)
COMP["provincia %"] = 100 * COMP["provincia kt CO2-eq"] / COMP["provincia kt CO2-eq"].sum()
COMP["nacional %"] = 100 * COMP["nacional kt CO2-eq"] / COMP["nacional kt CO2-eq"].sum()
COMP["participación en el país %"] = 100 * COMP["provincia kt CO2-eq"] / COMP["nacional kt CO2-eq"].where(COMP["nacional kt CO2-eq"] > 0)
display(COMP.style.format("{:,.1f}", na_rep="—"))
PC_NAC = sum(inv.NACIONAL_2022.values()) * 1000 / inv.POBLACION_NACIONAL
PC_NAC_SIN = (sum(inv.NACIONAL_2022.values()) - inv.NACIONAL_2022["4. USCUSS"]) * 1000 / inv.POBLACION_NACIONAL
print(f"Nacional por habitante: {PC_NAC:.2f} t CO2-eq/hab con USCUSS; {PC_NAC_SIN:.2f} sin USCUSS. Provincia: {TOTAL/POB:.2f}")
print("Ojo: el nacional incluye PIUP, industria y generación eléctrica, que este inventario no estima.")

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 5), gridspec_kw={"width_ratios": [1.1, 0.8, 1.3], "wspace": 0.75})
COLS = {"1. Energía": "#C2531F", "2. PIUP": "#8C8C8C", "3. Agricultura": "#6A994E", "4. USCUSS": "#386641", "5. Residuos": "#7F3C8D"}
ax = axs[0]
orden = [s for s in COLS if s in COMP.index]
x = np.arange(len(orden)); w = 0.38
ax.bar(x - w/2, COMP.loc[orden, "provincia %"], w, color=[COLS[s] for s in orden], label=PROVINCIA)
ax.bar(x + w/2, COMP.loc[orden, "nacional %"], w, color="white", edgecolor=[COLS[s] for s in orden], hatch="//", label="Ecuador 2022")
ax.set_xticks(x); ax.set_xticklabels([s.split(". ")[1] for s in orden], rotation=20)
ax.set_ylabel("% del total"); ax.set_title("a) Perfil por sector", loc="left", fontweight="bold"); ax.legend(frameon=False)
ax = axs[1]
ax.bar(GAS.index, GAS["t CO2-eq"] / 1000, color=["#444444", "#C2531F", "#2F6DB5"][:len(GAS)])
for i, (g, r) in enumerate(GAS.iterrows()): ax.text(i, r["t CO2-eq"] / 1000, f"{r['%']:.0f} %", ha="center", va="bottom")
ax.set_ylabel("kt CO2-eq"); ax.set_title("b) Por gas", loc="left", fontweight="bold")
ax = axs[2]
c = CLAVE.reset_index(); CORTO = {"1A3b": "Transporte", "1A4b": "Residencial (GLP)", "3A1": "Fermentación entérica", "3A2": "Estiércol",
         "3C7": "Arroz", "3C4": "Suelos, directo", "3C5": "Suelos, indirecto", "3B1b": "Deforestación",
         "4A": "Residuos sólidos", "4D1": "Aguas residuales"}
cod = c.categoria.str.split(" ").str[0]; c["etiqueta"] = cod + " " + cod.map(CORTO) + " · " + c.gas
c = c.iloc[::-1]
ax.barh(c.etiqueta, c["t CO2-eq"] / 1000, color=np.where(c.principal, "#C2531F", "#BBBBBB"))
ax.set_xlabel("kt CO2-eq"); ax.set_title("c) Categorías (en color, las principales)", loc="left", fontweight="bold")
for a in axs: a.spines[["top", "right"]].set_visible(False)
fig.suptitle(f"Inventario simplificado de GEI · {PROVINCIA} · {ANIO_INVENTARIO} · total {TOTAL/1000:,.0f} kt CO2-eq".replace(",", " ") + f" (± {U:.0f} %)", x=0.01, ha="left", fontweight="bold")
fig.text(0.01, -0.06, "IPCC 2006, Nivel 1; PCG del AR5. Nacional: INGEI 2022 (Segunda NDC del Ecuador). Elaboración propia con datos de actividad citados en la ficha.", fontsize=8.5, color="#666666")
fig.savefig(RES / f"IA2_01_inventario_{S}.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.show()

## 10. Ficha de datos de actividad y archivos para el capítulo 2

In [ ]:
FICHA = pd.DataFrame([
    ["Población", POB, "habitantes", "INEC, Censo de Población y Vivienda 2022"],
    ["Gasolina", GASOLINA_GAL, "galones", FUENTE_ENERGIA], ["Diésel", DIESEL_GAL, "galones", FUENTE_ENERGIA],
    ["GLP doméstico", GLP_KG, "kg", FUENTE_ENERGIA], ["Electricidad facturada", ELECTRICIDAD_MWH, "MWh", FUENTE_ENERGIA],
    ["Vacas de ordeño", VACAS_ORDENO, "cabezas", FUENTE_AGRICULTURA], ["Otros vacunos", OTROS_VACUNOS, "cabezas", FUENTE_AGRICULTURA],
    ["Porcinos", PORCINOS, "cabezas", FUENTE_AGRICULTURA], ["Ovinos", OVINOS, "cabezas", FUENTE_AGRICULTURA],
    ["Caprinos", CAPRINOS, "cabezas", FUENTE_AGRICULTURA], ["Caballos", CABALLOS, "cabezas", FUENTE_AGRICULTURA],
    ["Mulas y asnos", MULAS_ASNOS, "cabezas", FUENTE_AGRICULTURA], ["Arroz cosechado", ARROZ_HA, f"ha ({ARROZ_REGIMEN}, {ARROZ_DIAS} días)", FUENTE_AGRICULTURA],
    ["N sintético", N_SINTETICO_T, "t N", FUENTE_AGRICULTURA], ["Deforestación", DEFORESTACION_HA, "ha/año", FUENTE_USCUSS],
    ["Carbono del estrato", CARBONO_TC_HA, "t C/ha", FUENTE_USCUSS], ["Generación de residuos", GEN_KG_HAB_DIA, "kg/hab/día", FUENTE_RESIDUOS],
    ["Fracción dispuesta", FRAC_DISPUESTA, f"fracción ({SITIO})", FUENTE_RESIDUOS],
], columns=["dato", "valor", "unidad", "fuente"])
display(FICHA)
TAB.to_csv(RES / f"IA2_01_inventario_detalle_{S}.csv", index=False, encoding="utf-8")
SECT.round(1).to_csv(RES / f"IA2_01_resumen_sector_{S}.csv", encoding="utf-8")
CLAVE.round(2).to_csv(RES / f"IA2_01_categorias_principales_{S}.csv", encoding="utf-8")
COMP.round(2).to_csv(RES / f"IA2_01_comparacion_nacional_{S}.csv", encoding="utf-8")
FICHA.to_csv(RES / f"IA2_01_ficha_datos_actividad_{S}.csv", index=False, encoding="utf-8")
meta = {"material": "CC-IAM-P2-01 v1.0", "estudiante": ESTUDIANTE, "provincia": PROVINCIA, "anio": ANIO_INVENTARIO,
        "metodo": "IPCC 2006 Nivel 1; residuos sólidos por el método por defecto (balance de masa)",
        "pcg": inv.PCG, "pcg_fuente": inv.PCG_FUENTE, "total_tCO2e": round(TOTAL, 1), "incertidumbre_pct": round(U, 1),
        "completitud": COMPLETITUD.to_dict(), "nacional_fuente": inv.NACIONAL_FUENTE, "fecha": datetime.date.today().isoformat()}
(RES / "IA2_01_metadatos.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print(sorted(p.name for p in RES.glob("IA2_01_*")))

## 11. Preguntas para el capítulo 2

1. ¿Cuáles son las tres categorías principales de su provincia y qué porcentaje del total suman? ¿Se parecen al perfil nacional?
2. ¿Qué gas domina? Si domina el CH₄, explique de dónde viene y por qué su PCG cambia tanto el resultado (pruebe mentalmente con el PCG a 20 años, 81).
3. ¿Qué categorías quedaron como NE y cuánto podrían pesar? ¿Su total está subestimado o sobrestimado? Justifique.
4. ¿Cuál es la incertidumbre del total y qué categoría la domina? ¿Qué dato mejoraría primero para reducirla?
5. Compare sus emisiones por habitante con las del país. ¿Por qué no son directamente comparables?
6. La electricidad se reporta aparte (alcance 2): ¿por qué no se suma al total de la provincia?
7. ¿Qué sectores de su provincia serían prioritarios para mitigar? Esta respuesta alimenta la matriz de estrategias del paso 3.

**Para el informe:** año de los datos, fuente de cada dato de actividad, nivel del método (Nivel 1), PCG usados (AR5), categorías no estimadas y supuestos.

**Citas.** IPCC (2006). *Directrices del IPCC de 2006 para los inventarios nacionales de gases de efecto invernadero*, vols. 2, 4 y 5. IGES · IPCC (2013). *Cambio climático 2013: bases físicas*, cap. 8 (PCG) · MAATE (2025). *Segunda Contribución Determinada a Nivel Nacional del Ecuador* · CENACE (2025). *Factor de emisión de CO₂ del Sistema Nacional Interconectado, informe 2024* · INEC (2023). *Censo de Población y Vivienda 2022*.